#### 결측치 확인하기  
결측치는 데이터에서 값이 비어 있는 상태를 말한다.  
Pandas에서는 결측치를 NaN으로 표시하는 경우가 많음. NaN은 숫자 0이나 빈 문자열과 다르다. 값이 없다는 표시이므로 계산이나 조건 처리 전에 먼저 확인해야 한다. 전처리는 값을 바꾸거나 삭제하는 단계부터 시작하지 않는다. 먼저 어느 칼럼에 결측치가 있는지, 몇 개인지, 어떤 행에 포함되는지 파악해야 함.  
그래야 나중에 제거할지, 다른 값으로 채울지 판단할 수 있다.

In [4]:
import pandas as pd

df = pd.read_csv('./data/pandas_preprocess.csv')
print(df.shape)
print(df.head())

(122, 8)
  member_id   age gender signup_date     spend  visit_count   grade  memo
0    MB0001  29.0      F  2025/02/17  204200.0           10   BASIC    정상
1    MB0002  53.0      F  2025/02/17  980000.0            9   BASIC    정상
2    MB0003   NaN      F  2025/09/21   93300.0            9   BASIC  쿠폰사용
3    MB0004  58.0      F  2025/03/09  236900.0            8  SILVER    정상
4    MB0005  24.0      F  2025/08/01   64000.0           10   BASIC   NaN


In [5]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 122 entries, 0 to 121
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   member_id    122 non-null    object 
 1   age          119 non-null    float64
 2   gender       122 non-null    object 
 3   signup_date  122 non-null    object 
 4   spend        118 non-null    float64
 5   visit_count  122 non-null    int64  
 6   grade        122 non-null    object 
 7   memo         104 non-null    object 
dtypes: float64(2), int64(1), object(5)
memory usage: 7.8+ KB
None


In [6]:
print(df.isna().head())

   member_id    age  gender  signup_date  spend  visit_count  grade   memo
0      False  False   False        False  False        False  False  False
1      False  False   False        False  False        False  False  False
2      False   True   False        False  False        False  False  False
3      False  False   False        False  False        False  False  False
4      False  False   False        False  False        False  False   True


In [7]:
print(df.isna().sum())

member_id       0
age             3
gender          0
signup_date     0
spend           4
visit_count     0
grade           0
memo           18
dtype: int64


In [9]:
# 결측치가 있는 컬럼
missing = df.isna().sum()
print(type(missing))
print(missing[missing > 0])

<class 'pandas.core.series.Series'>
age       3
spend     4
memo     18
dtype: int64


In [10]:
# 전체 결측치 개수
print(df.isna().sum().sum())

25


In [ ]:
# 결측치가 있는 행 찾기
cond = df.isna().any(axis=1) # 여기서 axis=1은 행 기준으로 결측치가 하나라도 있으면 포함된다는 것!
print(cond.sum())
print(df.loc[cond, ['member_id', 'age', 'spend', 'memo']].head())

25
   member_id   age     spend  memo
2     MB0003   NaN   93300.0  쿠폰사용
4     MB0005  24.0   64000.0   NaN
5     MB0006  47.0       NaN  쿠폰사용
13    MB0014  32.0  254000.0   NaN
27    MB0028  30.0  182900.0   NaN


In [ ]:
# 특정 컬럼의 결측치
cond = df['spend'].isna()
print(df.loc[cond, 'member_id'].to_list()) # 'spend' 컬럼이 비어있는 member_id를 출력해서 리스트로 변환

['MB0006', 'MB0057', 'MB0083', 'MB0100']


#### 결측치 확인 시 주의사항  
1. 삭제와 대체 이전 단계  
결측치 확인은 삭제나 대체를 결정하기 전의 단계이다. age가 비어 있따고 해서 무조건 행을 삭제하는 것이 아님.  
분석 목적에 따라 평균이나 중앙값으로 채울 수도 있고, 해당 행을 제외할 수도 있다.  
2. 문자 칼럼의 결측치  
memo처럼 문자형 칼럼에 결측치가 있을 수도 있다. 숫자형 칼럼의 결측치는 평균이나 중앙값으로 대체하는 경우가 있지만, 문자형 칼럼은 '없음'같은 문자열로 채우거나 별도로 분류하는 방식도 가능하다. 이 절에서는 처리하지 않고 확인까지만 다룬다.  
3. 출력 확인  
결측치 개수를 확인한 뒤에는 필요한 컬럼만 선택해 실제 행을 살펴보면 좋다. 숫자만 보고 넘어가면 어떤 값이 비어 있는지 감이 잡히지 않는다.

#### 결측치 제거와 대체  
결측치를 확인했으니 다음 단계는 어떻게 처리할 것인가에 관한 것!  
결측치 처리는 크게 해당 행을 제외하는 방법과 빈 값을 다른 값으로 채우는 방법으로 나뉨  
어느 방법이 항상 정답이라고 할 수는 없으며, 문제의 요구 기준과 분석 목적, 컬럼의 성격에 따라 선택해야 함.  
빅분기 실기에서는 보통 결측치 제거하시오, 평균으로 대체하시오, 특정 값으로 채우시오 같은 처리 조건이 제시됨. 따라서 문제의 지시를 먼저 확인하고 그 조건에 맞는 코드를 정확히 작성하는게 중요!

1. 제거  
제거는 결측치가 있는 행이나 열을 분석 대상에서 제외하는 방식.  
행을 제거하면 계산에 사용하는 데이터 수가 줄어들기에 결측치가 매우 적거나, 해당 값이 없으면 문제 조건을 만족할 수 없는 경우에 사용하기 쉬움!  
2. 대체  
대체는 비어 있는 값을 다른 값으로 채우는 방식이다. 숫자형 컬럼은 평균, 중앙값, 0 등으로 채울 수 있고, 문자형 컬럼은 '기록없음'같은 문자열로 채울 수 있다. 대체를 사용하면 행 수를 유지할 수 있지만, 어떤 값으로 채웠는지에 따라 평균이난 합계가 달라질 수 있다!

#### 행 단위 제거  
1. 전체 결측치 제거  
dropna()는 결측치가 있는 행을 제거한다. 별도 조건을 주지 않는다면 결측치가 하나라도 있는 행은 무조건 제거됨

In [14]:
df_drop_all = df.dropna()

print(df.shape)
print(df_drop_all.shape)

(122, 8)
(97, 8)


2. 특정 컬럼 기준 제거  
특정 컬럼의 결측치만 기준으로 행을 제거할 수 있다.  
subset에 컬럼명을 지정하면 해당 컬럼들에 결측치가 있는 행만 제외함

In [ ]:
df_drop_key = df.dropna(subset=['age', 'spend']) # age와 spend 열에서 결측치를 가지는 행만 제거하는 방법
print(df_drop_key.shape)
print(df_drop_key[['member_id', 'age', 'spend', 'memo']].head(6))

(115, 8)
  member_id   age     spend  memo
0    MB0001  29.0  204200.0    정상
1    MB0002  53.0  980000.0    정상
3    MB0004  58.0  236900.0    정상
4    MB0005  24.0   64000.0   NaN
6    MB0007  43.0  129200.0  쿠폰사용
7    MB0008  29.0  297800.0   재방문


#### 값으로 대체  
1. 수치형 칼럼  
fillna()는 결측치를 지정한 값으로 채운다. 숫자형 칼럼에서는 평균이나 중앙값으로 대체하는 경우가 많다.  
다음 코드는 age의 결측치를 age 중앙값으로 채운다.

In [16]:
df_fill_age = df.copy()
median_age = df_fill_age['age'].median()

df_fill_age['age'] = df_fill_age['age'].fillna(median_age)

print(median_age)
print(df_fill_age.loc[df['age'].isna(), ['member_id', 'age']])

37.0
   member_id   age
2     MB0003  37.0
31    MB0032  37.0
77    MB0078  37.0


In [17]:
df_fill_memo = df.copy()
df_fill_memo['memo'] = df_fill_memo['memo'].fillna('기록없음')

print(df_fill_memo.loc[[4, 13, 27], ['member_id', 'memo']])

   member_id  memo
4     MB0005  기록없음
13    MB0014  기록없음
27    MB0028  기록없음


In [19]:
# 금액형 컬럼 - 지출액
# spend처럼 금액을 나타내는 컬럼도 문제 조건에 따라서 0, 평균, 중앙값 등으로 채울 수 있다. 여기서 중앙값으로 대체
df_fill_spend = df.copy()
median_spend = df_fill_spend['spend'].median()
df_fill_spend['spend'] = df_fill_spend['spend'].fillna(median_spend)

print(median_spend)
print(df_fill_spend.loc[df['spend'].isna(), ['member_id', 'spend']])

189450.0
   member_id     spend
5     MB0006  189450.0
56    MB0057  189450.0
82    MB0083  189450.0
99    MB0100  189450.0


In [21]:
# 처리 결과 확인
# 여러 컬럼의 결측치를 한 번에 처리할 때는 원본을 복사한 뒤 컬럼별로 다른 기준을 적용하는 방식이 안전하다.
df_clean = df.copy()

df_clean['age'] = df_clean['age'].fillna(df_clean['age'].median())
df_clean['spend'] = df_clean['spend'].fillna(df_clean['spend'].median())
df_clean['memo'] = df_clean['memo'].fillna('기록없음')

print(df_clean.isna().sum().sum())
print(df_clean.loc[[2,4,5,31], ['member_id', 'age', 'spend', 'memo']])

0
   member_id   age     spend  memo
2     MB0003  37.0   93300.0  쿠폰사용
4     MB0005  24.0   64000.0  기록없음
5     MB0006  47.0  189450.0  쿠폰사용
31    MB0032  37.0  301400.0    정상


#### 원본 보존  
결측치를 제거하거나 대체한 결과를 다시 사용하려면 새로운 변수에 저장하는 것이 좋다. 원본 df를 바로 바꾸면 앞 단계 결과와 비교하기 어려워질 수 있다. 실습 중에는 df_clean처럼 처리 결과를 담는 변수를 따로 두면 흐름을 확인하기 쉽다.

#### 이상치 확인  
이상치는 다른 값들과 비교했을때 유난히 작거나 큰 값을 말한다. 예를 들어 대부분의 지출액이 10만원에서 30만원 사이인데, 한 명만 98만원을 사용했다면 그 값을 먼저 확인해야 할 값들이 된다. 다만 이상치는 무조건 틀린 값이라는 뜻은 아님! 실제로 많이 구매한 일반적인 회원일 수도 있고, 입력 과정에 오류로 인해 0이 하나더 추가된 기록일 수도 있음! 전처리에서는 이상치를 바로 삭제하기 보단 먼저 확인한다. 